In [8]:
import os 
from dotenv import load_dotenv
from math import sqrt
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START , END, StateGraph
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel , Field
import time
from operator import add
from functools import reduce
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [2]:
class QuadraticState(TypedDict):
    a : int
    b : int 
    c : int
    equation : str
    disc : float
    result : str


In [18]:
def show_equation(state : QuadraticState):
    equation = f'{state["a"]}x2 + {state["b"]}x + {state["c"]}'
    return {"equation" : equation}

def calculate_disc(state : QuadraticState) :
    discriminant = (state["b"]**2) - 4*state["a"]*state["c"]
    return {"disc" : discriminant}

def real_roots(state: QuadraticState):
    root1 = round((-state["b"] + sqrt(state["disc"])) / (2* state["a"]), 2)
    root2 = round((-state["b"] - sqrt(state["disc"])) / (2* state["a"]), 2)
    result = f"The roots are root1 {root1} , and root2 {root2}"
    return {"result" : result}

def repeated_roots(state: QuadraticState):
    root = round(-state["b"] / (2* state["a"]), 2)
    resul = f"The repeated root is {root}"
    return {"result"  : resul}

def no_real_roots(state: QuadraticState):
    return {"result": "No real roots exists"}

# Routing Function
def check_condition(state : QuadraticState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:
    disc = state["disc"]
    if disc < 0:
        return "no_real_roots"
    elif disc == 0:
        return "repeated_roots"
    else:
        return "real_roots"
    

In [19]:
graph = StateGraph(QuadraticState)
graph.add_node("show_equation", show_equation)
graph.add_node("calculate_disc", calculate_disc)
graph.add_node("no_real_roots", no_real_roots)
graph.add_node("repeated_roots" , repeated_roots)
graph.add_node("real_roots", real_roots)

graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_disc")
graph.add_conditional_edges("calculate_disc", check_condition)
graph.add_edge("real_roots", END)
graph.add_edge("no_real_roots", END)
graph.add_edge("repeated_roots", END)
workflow = graph.compile()

In [22]:
initial_state = {"a" : 4, "b" : 4, "c" : 1}
workflow.invoke(initial_state)

{'a': 4,
 'b': 4,
 'c': 1,
 'equation': '4x2 + 4x + 1',
 'disc': 0,
 'result': 'The repeated root is -0.5'}